# Notebook 06 — Live Diagnostics Dashboard

This notebook shows how to use the LiveDashboard to monitor a running simulation.

## Metrics monitored
- Port status (which gRPC ports are occupied)
- ANSYS process PID, RAM, and CPU usage
- Current substep and NR iteration count
- Force and displacement residuals (convergence health)
- RST file size (disk usage)
- HFSS adaptive pass count (if running EM)

## How to use

```python
from ams.diagnostics.dashboard import LiveDashboard

# Start dashboard before the solve
db = LiveDashboard(mapdl, poll_interval_s=3.0, output_dir='outputs')
db.start()

# Run the solve (foreground)
mapdl.solve()

# Stop and get summary
db.stop()
db.print_summary()
db.export_csv()  # saves diagnostics.csv
```

## For a rich live terminal UI

```python
from ams.diagnostics.dashboard import live_solve_with_dashboard
from ams.mapdl.solver import SolverStrategy

strategy = SolverStrategy.from_config(cfg)
live_solve_with_dashboard(mapdl, strategy)
```

This requires the `rich` package (`pip install rich`) and displays a
live-updating table in the terminal.

In [ ]:
import sys; sys.path.insert(0, '..')

# ── Demonstrate port checking ──────────────────────────────────────────────
from ams.resources.manager import check_ports

status = check_ports()
print('Current port status:')
for port, occupied in status.items():
    icon = '🔴' if occupied else '🟢'
    print(f'  {icon} Port {port}: {"OCCUPIED" if occupied else "free"}')

In [ ]:
# ── Process health check ───────────────────────────────────────────────────
try:
    import psutil
    ansys_names = {'ansys252', 'ansys251', 'ansys242', 'ansysedt', 'mapdl'}
    found = []
    for proc in psutil.process_iter(['pid', 'name', 'memory_info', 'cpu_percent', 'status']):
        try:
            name = (proc.info['name'] or '').lower().replace('.exe', '')
            if name in ansys_names:
                mi = proc.info['memory_info']
                found.append({
                    'PID':   proc.info['pid'],
                    'Name':  proc.info['name'],
                    'RAM':   f"{mi.rss/1024**2:.0f} MB" if mi else '?',
                    'CPU%':  f"{proc.info['cpu_percent']:.1f}",
                })
        except Exception:
            pass
    if found:
        print(f'Found {len(found)} ANSYS processes:')
        for p in found:
            print(f'  PID {p["PID"]} | {p["Name"]} | RAM: {p["RAM"]} | CPU: {p["CPU%"]}%')
    else:
        print('No ANSYS processes running.')
except ImportError:
    print('psutil not installed. Run: pip install psutil')

In [ ]:
# ── Analyse a saved diagnostics log ───────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

diag_path = Path('../outputs/diagnostics.csv')

if diag_path.exists():
    df = pd.read_csv(diag_path)
    print(f'Loaded {len(df)} snapshots from {diag_path}')

    fig, axes = plt.subplots(2, 2, figsize=(12, 6))

    if 'f_residual' in df.columns:
        axes[0,0].semilogy(df['elapsed_s'], df['f_residual'].abs())
        axes[0,0].set_title('Force Residual'); axes[0,0].set_xlabel('Time (s)')
        axes[0,0].axhline(0.005, color='red', linestyle='--', label='ε_F=0.5%')
        axes[0,0].legend()

    if 'ram_mb' in df.columns:
        axes[0,1].plot(df['elapsed_s'], df['ram_mb'])
        axes[0,1].set_title('RAM Usage (MB)'); axes[0,1].set_xlabel('Time (s)')

    if 'substep' in df.columns:
        axes[1,0].step(df['elapsed_s'], df['substep'])
        axes[1,0].set_title('Substep Progress'); axes[1,0].set_xlabel('Time (s)')

    if 'rst_mb' in df.columns:
        axes[1,1].plot(df['elapsed_s'], df['rst_mb'])
        axes[1,1].set_title('RST File Size (MB)'); axes[1,1].set_xlabel('Time (s)')

    plt.tight_layout(); plt.show()
else:
    print(f'No diagnostics log found at {diag_path}')
    print('Run a simulation with LiveDashboard first, then re-run this cell.')

## Convergence Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Residual not decreasing | Bad element quality, BC error | Check mesh quality |
| Residual oscillating | Over-constrained or step too large | Increase substeps |
| Port 50052 occupied | Zombie MAPDL process | `kill_ansys_zombies()` |
| Diverged at substep 1 | Inverted elements or wrong material | Check Jacobian |
| Slow but converging | High aspect ratio elements | Refine mesh |
| RST file > 10 GB | Too many substeps stored | Use `OUTRES,LAST` |

**Next:** `07_results_and_visualization.ipynb`